<a href="https://colab.research.google.com/github/steveonyeke/python-ai-governance/blob/main/project-2-llm-evaluation-suite/01_environment_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 01: Environment and Baseline RAG Pipeline

**Goal:** Install and configure RAGAS, DeepEval, Langfuse, and Promptfoo, set up the Claude and Gemini API keys, and build a traced baseline RAG pipeline so every later phase has a live dashboard from its first run.

**Tools:** RAGAS, DeepEval, Langfuse v4, Promptfoo, Chroma, Gemini API (`google-genai`), Anthropic API (`anthropic`).

**Frameworks:** None directly evaluated in this phase. This phase establishes the observability layer that later phases use to demonstrate EU AI Act Article 12 record-keeping and NIST AI RMF Measure function alignment.

**Date:** July 2026

**Status:** In Progress

**System under test:** Gemini (`gemini-flash-latest`)
**Cross-model evaluator and judge throughout the project:** Claude (`claude-sonnet-4-6`)

In [2]:
# Cell 2: Drive mount and path setup

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/python-ai-governance-p2/data/"

import os
os.makedirs(DRIVE_PATH, exist_ok=True)

print("Drive mounted. Project 2 data path:", DRIVE_PATH)

Mounted at /content/drive
Drive mounted. Project 2 data path: /content/drive/MyDrive/python-ai-governance-p2/data/


In [3]:
# Cell 3: Installs and imports

!pip install -q ragas deepeval langfuse chromadb google-genai anthropic

# Promptfoo is a Node package, not Python. Installed here since Phase 1 sets up
# the full toolchain, even though Promptfoo is not used until Phase 5.
!npm install -g promptfoo

print("Install step complete. Restart runtime if prompted, then re-run from Cell 3.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 636.6/636.6 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 130.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [4]:
# API keys and secrets

from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = None

if ANTHROPIC_API_KEY:
  os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# Promptfoo config note (used in phase 5, set now so it is never forgotten later):
# this keeps all red-team generation local and avoids defaulting to OpenAI's API,
# since this project uses Claude and Gemini only.
os.environ['PROMPTFOO_DISABLE_REDTEAM_REMOTE_GENERATION'] = 'true'

print("Keys loaded. Gemini key present:", bool(GOOGLE_API_KEY), "/ Claude key present:", bool(ANTHROPIC_API_KEY))

Keys loaded. Gemini key present: True / Claude key present: False


In [5]:
# Client initializtion

from google import genai

gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
GEMINI_MODEL = "gemini-flash-latest"

claude_client = None
CLAUDE_MODEL = "claude-sonnet-4-6"

if ANTHROPIC_API_KEY:
  import anthropic
  claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Gemini client ready. Model:", GEMINI_MODEL)
print("Claude client ready:", claude_client is not None)

Gemini client ready. Model: gemini-flash-latest
Claude client ready: False


In [13]:
# Langfuse tracing setup
# Tracing goes in now, before any evaluation begins, not bolted on later.

from google.colab import userdata
from langfuse import Langfuse, observe

LANGFUSE_PUBLIC_KEY = userdata.get('LANGFUSE_PUBLIC_KEY')
LANGFUSE_SECRET_KEY = userdata.get('LANGFUSE_SECRET_KEY')
LANGFUSE_HOST = userdata.get('LANGFUSE_HOST')

os.environ['LANGFUSE_PUBLIC_KEY'] = LANGFUSE_PUBLIC_KEY
os.environ['LANGFUSE_SECRET_KEY'] = LANGFUSE_SECRET_KEY
os.environ['LANGFUSE_HOST'] = LANGFUSE_HOST

Langfuse = Langfuse()

# Sanity check: confirm the client can authenticate before building anything on top of it.
auth_ok = Langfuse.auth_check()
print("Langfuse authenticated:", auth_ok)

Langfuse authenticated: True


In [7]:
# Baseline document set
# A small, honest starting corpus. This is intentionally scoped and exploratory,
# not a production-scale knowledge base. Labeled here so later phases do not
# mistake this for a production corpus.

# EXPLORATORY: small governance-themed corpus for baseline testing only.
baseline_documents = [
    "The EU AI Act, Article 10, requires that training, validation, and testing data sets "
    "for high-risk AI systems be relevant, representative, and free of errors to the extent "
    "possible given the intended purpose.",
    "The NIST AI Risk Management Framework organizes AI governance activities into four "
    "functions: Govern, Map, Measure, and Manage. Measure covers testing, evaluation, "
    "verification, and validation activities.",
    "ISO/IEC 42001 is the first international management system standard for AI, specifying "
    "requirements for establishing, implementing, maintaining, and continually improving an "
    "AI management system within an organization.",
    "Nigeria's Data Protection Act 2023 (NDPA) established the Nigeria Data Protection "
    "Commission and introduced obligations for data controllers and processors that parallel "
    "several provisions of the GDPR.",
    "MITRE ATLAS is a knowledge base of adversary tactics and techniques against AI systems, "
    "modeled on the structure of the MITRE ATT&CK framework but scoped specifically to "
    "machine learning and AI system attack surfaces.",
]

print(f"Loaded {len(baseline_documents)} baseline documents for the exploratory corpus.")

Loaded 5 baseline documents for the exploratory corpus.


In [8]:
# Chroma vector store
# Same vector store choice as Project 1 Phase 9, carried forward for consistency.

import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="baseline_rag_phase1")

collection.add(
    documents=baseline_documents,
    ids=[f"doc_{i}" for i in range(len(baseline_documents))],
)

print("Chroma collection populated. Document count:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:09<00:00, 8.80MiB/s]


Chroma collection populated. Document count: 5


In [9]:
# Retrieval function, traced
# observe wraps this function so every retrieval call becomes a Langfuse span
# from the first run, rather than being added after the fact.

@observe(name="retrieve_context")
def retrieve_context(query, n_results=2):
    results = collection.query(query_texts=[query], n_results=n_results)
    return results["documents"][0]

In [10]:
# Generation function, traced
# Gemini is the system under test here. This is the pipeline that later phases
# will evaluate, not the evaluator itself.

@observe(name="generate_answer_gemini")
def generate_answer(query, context_chunks):
    context_block = "\n\n".join(context_chunks)
    prompt = (
        f"Answer the question using only the context provided. "
        f"If the context does not contain the answer, say so plainly.\n\n"
        f"Context:\n{context_block}\n\nQuestion: {query}"
    )
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
    )
    return response.text

In [11]:
# Full baseline RAG pipeline
# This is the pipeline Phase 2 onward will evaluate with RAGAS, DeepEval, and Promptfoo.

@observe(name="baseline_rag_pipeline")
def run_rag_pipeline(query):
    context_chunks = retrieve_context(query)
    answer = generate_answer(query, context_chunks)
    return {
        "query": query,
        "retrieved_context": context_chunks,
        "answer": answer,
    }

In [14]:
# Test run
# One sample query to confirm the pipeline runs end to end and produces a trace.

test_query = "What does the NIST AI Risk Management Framework require under the Measure function?"
result = run_rag_pipeline(test_query)

print("Query:", result["query"])
print("\nRetrieved context:")
for chunk in result["retrieved_context"]:
    print("-", chunk)
print("\nAnswer:")
print(result["answer"])

# Flush so the trace is visible in the Langfuse dashboard immediately, not just at process exit.
Langfuse.flush()

Query: What does the NIST AI Risk Management Framework require under the Measure function?

Retrieved context:
- The NIST AI Risk Management Framework organizes AI governance activities into four functions: Govern, Map, Measure, and Manage. Measure covers testing, evaluation, verification, and validation activities.
- The EU AI Act, Article 10, requires that training, validation, and testing data sets for high-risk AI systems be relevant, representative, and free of errors to the extent possible given the intended purpose.

Answer:
Based on the provided context, the NIST AI Risk Management Framework does not explicitly state what it *requires* under the Measure function; it only states that the Measure function "covers testing, evaluation, verification, and validation activities."


In [16]:
# Save baseline artifacts to Drive
# Persist the baseline corpus and the test result to DRIVE_PATH so Phase 02
# notebooks can load this exact baseline rather than rebuilding it from the scratch.

import json

with open(DRIVE_PATH + "phase01_baseline_documents.json", "w") as f:
    json.dump(baseline_documents, f, indent=2)

with open(DRIVE_PATH + "phase01_test_result.json", "w") as f:
    json.dump(result, f, indent=2)

print("Baseline artifacts saved to:", DRIVE_PATH)

Baseline artifacts saved to: /content/drive/MyDrive/python-ai-governance-p2/data/


In [17]:
# Cell 14: Promptfoo config scaffold (for Phase 05, written now so the file exists in the repo structure)
# Written to disk, not run yet. Phase 05 will populate the actual test cases and run this.

promptfoo_config = '''
description: "Phase 05 red-team scaffold, Production LLM Evaluation Suite"
providers:
  - id: google:gemini-flash-latest
prompts:
  - "{{query}}"
# Test cases and red-team plugin selection (OWASP LLM Top 10 2025,
# OWASP Top 10 for Agentic Applications, MITRE ATLAS v5.4.0 mapping)
# get added in Phase 05, not here.
'''

with open("promptfooconfig.yaml", "w") as f:
    f.write(promptfoo_config)

print("Promptfoo config scaffold written. This is a placeholder for Phase 05, not a working red-team config yet.")

Promptfoo config scaffold written. This is a placeholder for Phase 05, not a working red-team config yet.


## Findings

**What was built:** The full Project 2 toolchain (RAGAS, DeepEval, Langfuse, Promptfoo, Chroma), two separate model clients (Gemini as the system under test, Claude as the cross-model evaluator and judge, added later once Anthropic billing was set up), Langfuse v4 tracing wired into the retrieval, generation, and full pipeline functions before any evaluation work began, a small labeled exploratory baseline corpus in Chroma, and a traced baseline RAG pipeline tested end to end and persisted to Drive at `python-ai-governance-p2/data/`.

**What was found:** This phase produces no governance findings of its own. Its output is an operating, traced baseline. The one substantive design decision worth naming plainly is the enforced separation between the system under test and the judge model, decided before a single evaluation metric has been run. A secondary, practical finding from actually running this phase: Colab's runtime restarted the kernel more than once during setup, silently discarding prior cell state. This is a reminder that reproducibility in this environment depends on re-running cells in order after any restart, not assuming state persists.

**What it means:** Wiring observability in before evaluation begins, rather than after, is itself a governance-relevant choice. It means every future phase's numbers are traceable back to a specific run, not reconstructed after the fact. This directly supports EU AI Act Article 12 record-keeping expectations and the NIST AI RMF Measure function, both of which require verifiable evidence trails rather than retrospective claims. This traced pipeline is also the technical seed of what Afrispan Data Labs would eventually deliver to SME clients: a system where every model call is observable and auditable by design. Status: BUILDING, not yet a client-facing deliverable.

**Next step:** Phase 02a. Evaluate this baseline pipeline with RAGAS using Gemini as the configured LLM judge across faithfulness, answer relevancy, context precision, context recall, and noise sensitivity. Phase 02b repeats the same evaluation with Claude as judge and quantifies the score difference between the two runs.